In [15]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import plotly.express as px
# Import the processing module from the same folder
from processing import load_solutions, add_kwargs_as_indices, combine_solutions, read_parquet_and_convert, add_fields
# import processing 
# from pivottablejs import pivot_ui
G_save = True

In [16]:
def create_envelope(s_ed, s_uc, group_by = ['configuration', 'µ', 'iteration', 'day', 'hour,', 'r_id']):
    # Copy the relevant columns from s_ed['storage']
    envelope = s_ed['storage'][group_by + ['SOE_MWh', 'envelope_up_MWh', 'envelope_down_MWh']].copy()

    # Perform the first left join
    envelope = envelope.merge(
        s_uc['storage'][[col for col in group_by if col != 'iteration'] + ['SOE_MWh', 'envelope_up_MWh', 'envelope_down_MWh']].rename(
            columns={'SOE_MWh': 'SOE_DA_MWh', 'envelope_up_MWh': 'envelope_up_DA_MWh', 'envelope_down_MWh': 'envelope_down_DA_MWh'}
        ),
        on=[col for col in group_by if col != 'iteration'],
        how='left'
    )
    
    # Perform the second left join
    envelope = envelope.merge(
        s_uc['storage_parameters'][['r_id', 'SOE_max_MWh', 'initial_energy_proportion']].drop_duplicates(),
        on='r_id',
        how='left'
    )
    
    # Calculate initial state of energy (SOE) based on maximum SOE and initial energy proportion
    envelope['SOE_0_MWh'] = envelope['SOE_max_MWh'] * envelope['initial_energy_proportion']
    envelope['SOC'] = envelope['SOE_MWh'] / envelope['SOE_max_MWh'] 
    # Group by day, configuration, and resource ID, and get the last entry for each group

    return envelope

In [17]:

# ρs = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99]
# s_57_4 = [{'solution_folder': f"solutions_v57.{ρ}.4s", 'ρ': ρ, 'VLGEN': 30, 'model_type' : 'envelope'} for ρ in ρs]
# s_58_4 = [{'solution_folder': f"solutions_v58.{ρ}.4s", 'ρ': ρ, 'VLGEN': 1e-6,'model_type' : 'e-reserve'} for ρ in ρs]
# s_s57_7_7 = [{'solution_folder': f"solutions_v_s57.{ρ}.7.7s", 'ρ': ρ, 'VLGEN': 1000, 'model_type' : 'stochastic'} for ρ in ρs]
s_1_0 = [{'solution_folder': f"RTS-GMLC_v1.0s", 'VLGEN': 1e-6, 'model_type' : 'envelope'}]
s_1_0z = [{'solution_folder': f"RTS-GMLC_v1.0z", 'VLGEN': 1e-6, 'model_type' : 'envelope_bis'}]
s_2_0 = [{'solution_folder': f"RTS-GMLC_v2.0s", 'VLGEN': 1e-6, 'model_type' : 'e-reserve'}]
s_s1_0 = [{'solution_folder': f"RTS-GMLC_v_s1.0s", 'VLGEN': 1e-6, 'model_type' : 'stochastic'}]

s_uc = []
s_ed = []
gcd_KPI_adequacy = []
gcdi_KPI_adequacy = []
# ss = s_57_4 + s_58_4 + s_s57_7_7
ss = s_1_0 + s_2_0 + s_s1_0
ss = s_1_0 + s_1_0z
for sol in ss:
    # ρ = sol['ρ']
    s = sol['solution_folder']
    s_uc_name = 's_suc' if sol['model_type'] == 'stochastic' else 's_uc'
    # s_uc_ = load_solutions(s_uc_name, os.path.join("..", "output", s), [7], model_type = sol['model_type'], VLGEN=sol['VLGEN'], ρ=ρ, solution_id = s)
    # s_ed_ = load_solutions("s_ed", os.path.join("..", "output", s), [7], model_type = sol['model_type'], VLGEN=sol['VLGEN'], ρ=ρ, solution_id = s)
    # s_uc.append(s_uc_)
    # s_ed.append(s_ed_)

    gcd_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcd_KPI_adequacy.parquet"))
    gcd_KPI_adequacy_ = add_fields(gcd_KPI_adequacy_, model_type = sol['model_type'], VLGEN=sol['VLGEN'], solution_id = s) 

    gcdi_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcdi_KPI_adequacy.parquet"))
    gcdi_KPI_adequacy_ = add_fields(gcdi_KPI_adequacy_, model_type = sol['model_type'], VLGEN=sol['VLGEN'], solution_id = s)

    gcd_KPI_adequacy.append(gcd_KPI_adequacy_)
    gcdi_KPI_adequacy.append(gcdi_KPI_adequacy_)

# s_uc = combine_solutions(s_uc)
# s_ed = combine_solutions(s_ed)
gcd_KPI_adequacy = pd.concat(gcd_KPI_adequacy)
gcdi_KPI_adequacy = pd.concat(gcdi_KPI_adequacy)

if 'µ' in gcdi_KPI_adequacy.columns: 
#         # out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)
    gcdi_KPI_adequacy['model_type'] = gcdi_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
    gcd_KPI_adequacy['model_type'] = gcd_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)




../output/RTS-GMLC_v1.0s/all_gcd_KPI_adequacy.parquet
../output/RTS-GMLC_v1.0s/all_gcdi_KPI_adequacy.parquet
../output/RTS-GMLC_v1.0z/all_gcd_KPI_adequacy.parquet
../output/RTS-GMLC_v1.0z/all_gcdi_KPI_adequacy.parquet


In [18]:
gcdi_KPI_adequacy['model_type'].unique()

array(['envelope', 'conservative', 'envelope_bis'], dtype=object)

In [19]:
if G_save:
    out= gcd_KPI_adequacy.copy()
    if 'µ' in out.columns: 
        out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)
        # out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: ((x[0] =='envelope')*(x[1]==1)*'conservatice' + x[0]), axis = 1)

    renames = {'µ': 'mu', 'ρ' : 'rho'}
    renames = {k: v for k, v in renames.items() if k in out.columns}
    out.rename(columns = renames, inplace = True)
    out.to_csv('gcd_KPI_adequacy.csv', index=False)
    gcdi_KPI_adequacy.rename(columns = renames, inplace = True)
    gcdi_KPI_adequacy.reset_index().to_csv('gcdi_KPI_adequacy.csv', index=False)

/tmp/ipykernel_42931/3797213802.py:4: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



In [20]:
import plotly.graph_objects as go

data = [
    go.Bar(
        x=['Q1', 'Q2', 'Q3', 'Q4'],
        y=[150, 200, 250, 300],
        name='New York',
        offsetgroup="USA"
    ),
    go.Bar(
        x=['Q1', 'Q2', 'Q3', 'Q4'],
        y=[180, 220, 270, 320],
        name='Boston',
        offsetgroup="USA"
    ),
    go.Bar(
        x=['Q1', 'Q2', 'Q3', 'Q4'],
        y=[130, 170, 210, 260],
        name='Montreal',
        offsetgroup="Canada"
    ),
    go.Bar(
        x=['Q1', 'Q2', 'Q3', 'Q4'],
        y=[160, 210, 260, 310],
        name='Toronto',
        offsetgroup="Canada"
    )
]

layout = go.Layout(
    title={
        'text': 'Quarterly Sales by City, Grouped by Country'
    },
    xaxis={
        'title': {
            'text': 'Quarter'
        }
    },
    yaxis={
        'title': {
            'text': 'Sales'
        }
    },
    barmode='stack'
)

fig = go.Figure(data=data, layout=layout)

fig.show()


In [21]:

# (aux == gcdi_KPI_adequacy.iloc[1]).all()

In [22]:
aux3

NameError: name 'aux3' is not defined

In [ ]:
aux2

## Getting cost for comparison

## Saving SOC[s,T] for SUC

In [ ]:
# group_by =['solution_id','VLGEN', 'ρ','configuration', 'µ', 'iteration', 'day', 'hour', 'r_id', ]
# envelope = create_envelope(s_ed, s_uc, group_by = group_by) 
# group_by =['solution_id', 'VLGEN', 'ρ', 'configuration', 'µ', 'iteration', 'day', 'r_id']
# envelope_last = envelope.sort_values(['hour']).groupby(group_by).last()
# group_by =['solution_id', 'VLGEN', 'ρ', 'configuration', 'µ', 'day', 'r_id']
# # numeric_columns = envelope_last.select_dtypes(include='number').columns
# envelope_last_mean = envelope_last.groupby(group_by).mean()['SOC'].reset_index()
# envelope_last_mean = envelope_last_mean[envelope_last_mean['µ'] != 1] # We just want optimal µ
# # envelope_last_mean
# G_save = False
# if G_save: envelope_last_mean.reset_index().to_excel('envelope_last_mean.xlsx', index=False)

Put in reading mode

In [ ]:

# merged_df.sort_values(by='µ', inplace=True)
# Display the resulting dataframe


fig = px.histogram(ΔE[ΔE['VLGEN'] ==0], x='AE_MWh', facet_col ='VLGEN', color = 'AE_+')
fig.update_layout(xaxis_title='AE_MWh', yaxis_title='Frequency', barmode='overlay')
fig.update_traces(opacity=0.75)
fig.show()


In [ ]:
fig = px.histogram(ΔE, x='ΔSOE_MWh', facet_col ='VLGEN', color = 'AE_+')
fig.update_layout(xaxis_title='ΔSOE_MWh', yaxis_title='Frequency', barmode='overlay')
fig.update_traces(opacity=0.75)
fig.show()

In [ ]:
ΔE['ΔSOE_MWh'].groupby(ΔE['VLGEN']).describe()

In [ ]:


fig = px.histogram(ΔE, x='ΔE_MWh', facet_col ='VLGEN', color = 'AE_+')
fig.update_layout(xaxis_title='ΔE_MWh', yaxis_title='Frequency', barmode='overlay')
fig.update_traces(opacity=0.75)
fig.show()

In [ ]:
fig = px.box(ΔE, x='ρ', y='ΔSOE_MWh', color='VLGEN')
fig.update_layout(xaxis_title='ρ', yaxis_title='ΔSOE_MWh')
fig.show()
